# 🎯 Question Answering with HuggingFace Transformers: A Complete Tutorial

This notebook demonstrates **extractive question answering** using the SQuAD dataset and DistilBERT. Question answering is a fundamental NLP task where given a context paragraph and a question, the model must extract the answer span from the context.

## 📋 What you'll learn:
1. **Dataset Analysis**: Understanding SQuAD structure and answer formats
2. **Tokenization Challenges**: Handling long contexts with stride and overflow
3. **Answer Alignment**: Converting character positions to token indices
4. **Model Training**: Fine-tuning DistilBERT for question answering
5. **Evaluation**: Computing metrics and extracting answers from logits

## 🧠 Key Concepts:
- **Extractive QA**: Answers are spans directly extracted from the context
- **Token Alignment**: Mapping between character positions and token indices
- **Offset Mapping**: Understanding token-to-character relationships
- **Overflow Handling**: Managing contexts longer than model's max length

---

## 1. Load Dataset

In [1]:
from datasets import load_dataset

data=load_dataset("squad")
data

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

### 📊 Loading the SQuAD Dataset

**SQuAD (Stanford Question Answering Dataset)** is a reading comprehension dataset where:
- Each example contains a **context paragraph**, a **question**, and one or more **answers**
- Answers are **spans** from the context with their **character start positions**
- Training set has ~87k examples, validation set has ~10k examples

The dataset structure:
- `context`: The paragraph containing the answer
- `question`: The question to be answered
- `answers`: Dictionary with `text` (answer string) and `answer_start` (character position)

In [2]:
data["train"]["title"][0]

'University_of_Notre_Dame'

In [3]:
data["train"]["context"][0]


'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.'

In [4]:
data["train"]["question"][0]

'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?'

In [5]:
data["train"]["answers"][0]

{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}

### 🔍 Exploring Dataset Structure

Let's examine the structure of our data to understand what we're working with:

There might be more than one answer, but in train we dont see more than one. However, in case of validation, there are more than one

In [6]:
data["validation"]["answers"][2]

{'text': ['Santa Clara, California',
  "Levi's Stadium",
  "Levi's Stadium in the San Francisco Bay Area at Santa Clara, California."],
 'answer_start': [403, 355, 355]}

Sometimes all the answers can be same

In [7]:
data["validation"]["answers"][0]

{'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'],
 'answer_start': [177, 177, 177]}

#### 📝 Key Observation: Multiple Answers in Validation

**Important difference between train and validation:**
- **Training set**: Usually has only 1 answer per question
- **Validation set**: Often has multiple valid answers (from different annotators)

This is crucial for evaluation - we need to compare our prediction against all possible correct answers!

# 2. Tokenizer Initialize & Understanding BERT Input Format

### 🔧 Why DistilBERT for Question Answering?

**DistilBERT** is perfect for QA because:
- **Faster**: 60% smaller than BERT, 40% faster inference
- **Dual Input**: Can process question + context together
- **Token Classification**: Predicts start/end positions for answer spans

### 📝 BERT Input Format for QA:
```
[CLS] question tokens [SEP] context tokens [SEP]
```

The model learns to:
1. **Start logits**: Probability that each token is the START of the answer
2. **End logits**: Probability that each token is the END of the answer

In [8]:
from transformers import AutoTokenizer

tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

C:\Users\Lenovo\AppData\Roaming\Python\Python311\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
data["train"]["context"][4]

'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.'

In [10]:
context = data["train"]["context"][1]
question = data["train"]["question"][1]

In [11]:
question

'What is in front of the Notre Dame Main Building?'

In [12]:
inputs =tokenizer(question,context)
tokenizer.decode(inputs["input_ids"])

'[CLS] what is in front of the notre dame main building? [SEP] architecturally, the school has a catholic character. atop the main building\'s gold dome is a golden statue of the virgin mary. immediately in front of the main building and facing it, is a copper statue of christ with arms upraised with the legend " venite ad me omnes ". next to the main building is the basilica of the sacred heart. immediately behind the basilica is the grotto, a marian place of prayer and reflection. it is a replica of the grotto at lourdes, france where the virgin mary reputedly appeared to saint bernadette soubirous in 1858. at the end of the main drive ( and in a direct line that connects through 3 statues and the gold dome ), is a simple, modern stone statue of mary. [SEP]'

#### 🎯 Basic Tokenization Example

Let's see how the tokenizer combines question and context:

# Token limit problem

### ⚠️ The Token Limit Challenge

**BERT's constraint**: Maximum 512 tokens input length

**Real-world problem**: Many context paragraphs are much longer than 512 tokens!

**Solution Strategy**:
1. **Truncation**: `truncation="only_second"` - only truncate context, keep full question
2. **Sliding Window**: Use `stride` to create overlapping chunks
3. **Overflow Tokens**: `return_overflowing_tokens=True` creates multiple inputs from one long context

**Why this works**: The answer might be in any chunk, so we process all chunks and find the best answer across all of them.

In [13]:
inputs=tokenizer(
    question,
    context,
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True
)

#### 🔄 Stride & Overflow Demo

Let's see how **stride=50** creates overlapping chunks with **max_length=100**:

**Parameters explained**:
- `max_length=100`: Each chunk has max 100 tokens
- `stride=50`: Overlap of 50 tokens between consecutive chunks
- `return_overflowing_tokens=True`: Create multiple inputs from long text

In [14]:
for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))

[CLS] what is in front of the notre dame main building? [SEP] architecturally, the school has a catholic character. atop the main building's gold dome is a golden statue of the virgin mary. immediately in front of the main building and facing it, is a copper statue of christ with arms upraised with the legend " venite ad me omnes ". next to the main building is the basilica of the sacred heart. immediately behind the basilica is the grotto, a marian [SEP]
[CLS] what is in front of the notre dame main building? [SEP] it, is a copper statue of christ with arms upraised with the legend " venite ad me omnes ". next to the main building is the basilica of the sacred heart. immediately behind the basilica is the grotto, a marian place of prayer and reflection. it is a replica of the grotto at lourdes, france where the virgin mary reputedly appeared to saint bernadette soubirous in [SEP]
[CLS] what is in front of the notre dame main building? [SEP] sacred heart. immediately behind the basilic

In [15]:
inputs.keys()

dict_keys(['input_ids', 'attention_mask', 'overflow_to_sample_mapping'])

In [16]:
inputs["overflow_to_sample_mapping"]

[0, 0, 0]

#### 🗂️ Understanding Overflow Mapping

**`overflow_to_sample_mapping`** is crucial! It tells us which original sample each chunk belongs to.

Example: `[0, 0, 1, 1]` means:
- Chunks 0,1 come from original sample 0
- Chunks 2,3 come from original sample 1

This is essential when processing batches - we need to know which answer corresponds to which chunk!

In [17]:
inputs=tokenizer(
    data["train"]["question"][:3],
    data["train"]["context"][:3],
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True 
)
inputs["overflow_to_sample_mapping"]

[0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 2]

In [18]:
inputs["offset_mapping"]

[[(0, 0),
  (0, 2),
  (3, 7),
  (8, 11),
  (12, 15),
  (16, 22),
  (23, 27),
  (28, 37),
  (38, 44),
  (45, 47),
  (48, 52),
  (53, 55),
  (56, 59),
  (59, 63),
  (64, 70),
  (70, 71),
  (0, 0),
  (0, 13),
  (13, 15),
  (15, 16),
  (17, 20),
  (21, 27),
  (28, 31),
  (32, 33),
  (34, 42),
  (43, 52),
  (52, 53),
  (54, 58),
  (59, 62),
  (63, 67),
  (68, 76),
  (76, 77),
  (77, 78),
  (79, 83),
  (84, 88),
  (89, 91),
  (92, 93),
  (94, 100),
  (101, 107),
  (108, 110),
  (111, 114),
  (115, 121),
  (122, 126),
  (126, 127),
  (128, 139),
  (140, 142),
  (143, 148),
  (149, 151),
  (152, 155),
  (156, 160),
  (161, 169),
  (170, 173),
  (174, 180),
  (181, 183),
  (183, 184),
  (185, 187),
  (188, 189),
  (190, 196),
  (197, 203),
  (204, 206),
  (207, 213),
  (214, 218),
  (219, 223),
  (224, 226),
  (226, 229),
  (229, 232),
  (233, 237),
  (238, 241),
  (242, 248),
  (249, 250),
  (250, 252),
  (252, 254),
  (254, 256),
  (257, 259),
  (260, 262),
  (263, 265),
  (265, 268),
  (268,

#### 📍 Offset Mapping: The Bridge Between Tokens and Characters

**`offset_mapping`** is the **most critical concept** for question answering!

Each token gets a tuple `(start_char, end_char)` showing its position in the original text.

**Why we need this**:
- Dataset gives answer positions as **character indices**
- Model works with **token indices**
- Offset mapping translates between them!

Example: `(45, 52)` means this token spans characters 45-52 in the original text.

In [19]:
for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))

[CLS] to whom did the virgin mary allegedly appear in 1858 in lourdes france? [SEP] architecturally, the school has a catholic character. atop the main building's gold dome is a golden statue of the virgin mary. immediately in front of the main building and facing it, is a copper statue of christ with arms upraised with the legend " venite ad me omnes ". next to the main building is the basilica of the sacred heart. immediately behind the basilica is the gr [SEP]
[CLS] to whom did the virgin mary allegedly appear in 1858 in lourdes france? [SEP] main building and facing it, is a copper statue of christ with arms upraised with the legend " venite ad me omnes ". next to the main building is the basilica of the sacred heart. immediately behind the basilica is the grotto, a marian place of prayer and reflection. it is a replica of the grotto at lourdes, france where the virgin mary reputedly appeared to saint [SEP]
[CLS] to whom did the virgin mary allegedly appear in 1858 in lourdes franc

size of the offset mapping list, they are splitted into 4 inputs

In [20]:
len(inputs["input_ids"])

11

In [21]:
inputs.keys()

dict_keys(['input_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping'])

In [22]:
len(inputs["offset_mapping"])

11

In [23]:
len(inputs["overflow_to_sample_mapping"])

11

# 3. Aligning the Target: Converting Character Positions to Token Indices

### 🎯 The Core Challenge

**The Problem**: 
- SQuAD gives answers as **character positions** in original text
- Models need **token positions** for training

**The Solution**:
1. Find where context starts/ends in tokenized sequence
2. Use offset mapping to locate answer within context tokens
3. Convert character positions to token indices

### 🔍 Sequence IDs Explanation

`sequence_ids()` returns:
- `None`: Special tokens (CLS, SEP)
- `0`: Question tokens  
- `1`: Context tokens

This helps us identify which tokens belong to the context (where answers can be found).

In [24]:
print(inputs.sequence_ids())

[None, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, None, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, None]


<img src="st_en_finder.png" alt="alt text" width="1000" height="600">

### 📊 Visual Guide: Token Sequence Structure

The image below shows how BERT processes question + context and how we identify answer positions:


In [25]:
data["train"][1]

{'id': '5733be284776f4190066117f',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'What is in front of the Notre Dame Main Building?',
 'answers': {'text': ['a copper statue of Christ'], 'answer_start': [188]}}

In [26]:
answer=data["train"][1]["answers"]
answer

{'text': ['a copper statue of Christ'], 'answer_start': [188]}

In [27]:
sequence_ids = inputs.sequence_ids(0)
context_start = sequence_ids.index(1)
context_end = len(sequence_ids) - sequence_ids[::-1].index(1) - 1
context_start, context_end

(17, 98)

#### 🎯 Step 1: Find Context Boundaries

We need to locate where the context starts and ends in our tokenized sequence:

- `context_start`: First position where sequence_id = 1 (first context token)
- `context_end`: Last position where sequence_id = 1 (last context token)

**Why this matters**: Answers can only exist within the context portion, never in question or special tokens!

In [28]:
inputs["overflow_to_sample_mapping"]

[0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 2]

In [29]:
ans_start_char=answer["answer_start"][0]
ans_start_char

188

In [30]:
ans_end_char=ans_start_char + len(answer["text"][0])
ans_end_char

213

#### 📍 Step 2: Get Answer Character Positions

From the dataset, we extract:
- `ans_start_char`: Character position where answer begins
- `ans_end_char`: Character position where answer ends

These are **character indices** in the original context text.

In [31]:
offset_mapping = inputs["offset_mapping"][0]

In [32]:
print(offset_mapping[:100])

[(0, 0), (0, 2), (3, 7), (8, 11), (12, 15), (16, 22), (23, 27), (28, 37), (38, 44), (45, 47), (48, 52), (53, 55), (56, 59), (59, 63), (64, 70), (70, 71), (0, 0), (0, 13), (13, 15), (15, 16), (17, 20), (21, 27), (28, 31), (32, 33), (34, 42), (43, 52), (52, 53), (54, 58), (59, 62), (63, 67), (68, 76), (76, 77), (77, 78), (79, 83), (84, 88), (89, 91), (92, 93), (94, 100), (101, 107), (108, 110), (111, 114), (115, 121), (122, 126), (126, 127), (128, 139), (140, 142), (143, 148), (149, 151), (152, 155), (156, 160), (161, 169), (170, 173), (174, 180), (181, 183), (183, 184), (185, 187), (188, 189), (190, 196), (197, 203), (204, 206), (207, 213), (214, 218), (219, 223), (224, 226), (226, 229), (229, 232), (233, 237), (238, 241), (242, 248), (249, 250), (250, 252), (252, 254), (254, 256), (257, 259), (260, 262), (263, 265), (265, 268), (268, 269), (269, 270), (271, 275), (276, 278), (279, 282), (283, 287), (288, 296), (297, 299), (300, 303), (304, 312), (313, 315), (316, 319), (320, 326), (327

Find the first 1

In [33]:
offset_mapping[context_start]

(0, 13)

Find the last 1

In [34]:
offset_mapping[context_end]

(373, 375)

#### 🔍 Step 3: The Critical Alignment Process

**The Big Challenge**: Convert character positions → token indices

**Our strategy**:
1. **Check if answer exists in this chunk**: Answer chars must fall within context boundaries
2. **Search through offset mappings**: Find tokens whose char ranges match answer positions
3. **Extract token indices**: Get start_idx and end_idx for the answer span

**Edge Case**: If answer doesn't fit in current chunk → set indices to 0 (no answer)

In [35]:
start_idx=0
end_idx=0 # since we dont know the start and end index of the answer in the context

#If the answer is not in the context, we will not be able to find the start and end index
if(offset_mapping[context_start][0] > ans_start_char or offset_mapping[context_end][1] < ans_end_char):
# the offset_mapping[context_start][0] is the start character of the context
# the offset_mapping[context_end][1] is the end character of the context
# That means if the start character of the context is greater than the start character of the answer or the end character of the context is less than the end character of the answer, then the answer is not in the context
#or simply not in this offset_mapping
    print("Answer is not in the context")
else:
    i=context_start
    for start_end_char in offset_mapping[context_start:]: 
        start,end=start_end_char # start_end_char is a tuple of (start, end) character of the context
        if( start == ans_start_char):# at some point, the start character of the context will be equal to the start character of the answer
            start_idx=i
        if( end == ans_end_char): # at some point, the end character of the context will be equal to the end character of the answer
            end_idx=i
            break
        i+=1

start_idx, end_idx

(56, 60)

In [36]:
input_ids=inputs["input_ids"][0]
result=input_ids[start_idx:end_idx+1]
result

[1037, 6967, 6231, 1997, 4828]

#### ✅ Step 4: Verification - Did We Get It Right?

Let's verify our token extraction by decoding the tokens back to text:

In [37]:
tokenizer.decode(result)

'a copper statue of christ'

In [38]:
answer

{'text': ['a copper statue of Christ'], 'answer_start': [188]}

In [39]:
context_start,context_end,ans_start_char,ans_end_char

(17, 98, 188, 213)

In [40]:
def find_answer_token_index(context_start,context_end,ans_start_char,ans_end_char,offset_mapping):
    start_idx=0
    end_idx=0
 
    if(offset_mapping[context_start][0]>ans_start_char or offset_mapping[context_end][1]<ans_end_char):
        pass
    else:
        i=context_start
        for start_end_char in offset_mapping[context_start:]:
            start,end=start_end_char
            if( start == ans_start_char):
                start_idx=i
            if( end == ans_end_char):
                end_idx=i
                break
            i+=1
    return start_idx,end_idx

#### 🛠️ Creating a Reusable Function

Now let's package our logic into a reusable function that handles the answer alignment:

In [41]:
print(inputs["offset_mapping"])

[[(0, 0), (0, 2), (3, 7), (8, 11), (12, 15), (16, 22), (23, 27), (28, 37), (38, 44), (45, 47), (48, 52), (53, 55), (56, 59), (59, 63), (64, 70), (70, 71), (0, 0), (0, 13), (13, 15), (15, 16), (17, 20), (21, 27), (28, 31), (32, 33), (34, 42), (43, 52), (52, 53), (54, 58), (59, 62), (63, 67), (68, 76), (76, 77), (77, 78), (79, 83), (84, 88), (89, 91), (92, 93), (94, 100), (101, 107), (108, 110), (111, 114), (115, 121), (122, 126), (126, 127), (128, 139), (140, 142), (143, 148), (149, 151), (152, 155), (156, 160), (161, 169), (170, 173), (174, 180), (181, 183), (183, 184), (185, 187), (188, 189), (190, 196), (197, 203), (204, 206), (207, 213), (214, 218), (219, 223), (224, 226), (226, 229), (229, 232), (233, 237), (238, 241), (242, 248), (249, 250), (250, 252), (252, 254), (254, 256), (257, 259), (260, 262), (263, 265), (265, 268), (268, 269), (269, 270), (271, 275), (276, 278), (279, 282), (283, 287), (288, 296), (297, 299), (300, 303), (304, 312), (313, 315), (316, 319), (320, 326), (32

In [42]:
# CORRECTED: Create separate lists for start and end indices
start_indices = []  # Separate list for start positions
end_indices = []    # Separate list for end positions

for i, offset_mapping in enumerate(inputs["offset_mapping"]):
    sequence_ids = inputs.sequence_ids(i)

    context_start = sequence_ids.index(1)
    context_end = len(sequence_ids) - sequence_ids[::-1].index(1) - 1

    start_idx, end_idx = find_answer_token_index(
        context_start,
        context_end,
        ans_start_char,
        ans_end_char,
        offset_mapping
    )
    start_indices.append(start_idx)
    end_indices.append(end_idx)

#### 🔄 Processing Multiple Chunks

When we have multiple chunks from overflow, we need to:
1. Process each chunk separately
2. Find answer positions for each chunk (if answer exists in that chunk)
3. Store results in separate lists

**Critical Bug Fix**: We must create separate lists for start and end indices!

In [43]:
# Display the corrected results
print("Start indices:", start_indices)
print("End indices:", end_indices)
# The result shows if there are 0 /-1 in the start and end indices, which means the answer is not in the context

Start indices: [56, 24, 0, 0, 52, 16, 0, 56, 24, 0, 0]
End indices: [60, 28, 0, 0, 56, 20, 0, 60, 28, 0, 0]


In [44]:
# Check for any questions that have leading or trailing whitespace
for q in data["train"]["question"][:1000]:
    if(q.strip() != q):
        print(q)

In what city and state did Beyonce  grow up? 
 The album, Dangerously in Love  achieved what spot on the Billboard Top 100 chart?
Which song did Beyonce sing at the first couple's inaugural ball? 
What event did Beyoncé perform at one month after Obama's inauguration? 
Where was the album released? 
What movie influenced Beyonce towards empowerment themes? 


# 4. Writing the Tokenize Function: Putting It All Together

### 🏗️ Building the Complete Tokenization Pipeline

Now we'll create a comprehensive function that:
1. **Tokenizes** question-context pairs with proper overflow handling
2. **Aligns** answers from character positions to token indices  
3. **Handles** multiple samples in batches efficiently
4. **Prepares** data in the exact format needed for model training

**Key Parameters**:
- `max_length=384`: Standard for QA (Google's choice)
- `stride=128`: Good balance between overlap and efficiency

In [45]:
data["train"]["question"][:10]

['To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'What is in front of the Notre Dame Main Building?',
 'The Basilica of the Sacred heart at Notre Dame is beside to which structure?',
 'What is the Grotto at Notre Dame?',
 'What sits on top of the Main Building at Notre Dame?',
 'When did the Scholastic Magazine of Notre dame begin publishing?',
 "How often is Notre Dame's the Juggler published?",
 'What is the daily student paper at Notre Dame called?',
 'How many student news papers are found at Notre Dame?',
 'In what year did the student paper Common Sense begin publication at Notre Dame?']


---

### 🔍 **Understanding `overflow_to_sample_mapping` & `answer_start` in SQuAD**

When we tokenize long paragraphs for SQuAD, the tokenizer **splits them into overlapping chunks** (because models can only handle limited tokens).

👉 Example split:

```
Original paragraph: "Dhaka is the capital of Bangladesh. It has many rivers and a busy port."
Chunks:  
  0 → characters 0–150  
  1 → characters 100–250  
  2 → characters 200–300
```

---
### 🟢 **What is `overflow_to_sample_mapping`?**

* Each chunk needs to “remember” **which original sample** it came from.
* Example:

  ```
  [0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 2]
  ```

  ✅ Chunks 0–3 belong to **sample 0**
  ✅ Chunks 4–6 belong to **sample 1**
  ✅ Chunks 7–10 belong to **sample 2**

---

### 🔵 **What is `answer_start`?**

* In the dataset, the answer is marked **by character position** in the **original text**.
* Example:

  ```
  answer['text'] = ["Bangladesh"]
  answer['answer_start'] = [213]
  ```

  👉 Means the word **“Bangladesh” starts at character 213**.

---

### 🟣 **Why loop over chunks?**

Each chunk must check if it **contains the answer**:

```python
for i, offset in enumerate(offset_mapping):
    sample_idx = orig_sample_idxs[i]    # which original paragraph this chunk came from
    answer = answers[sample_idx]        # grab that answer
    ans_start_char = answer['answer_start'][0]
```

✅ This makes sure the code always looks up the **right answer** for each chunk.

---

### 🏗️ **How does it find the token index?**

* Each chunk has an `offset_mapping` → tells you **(start\_char, end\_char)** for every token.
* The code compares:

  * `ans_start_char` (e.g., 213)
  * `ans_end_char` (213 + len(“Bangladesh”))

➡ If these character positions fall inside this chunk’s offsets → **find the token index** where the answer starts and ends.
➡ If not → mark this chunk as “no answer.”

---

### ✅ **Mini Example**

* Answer starts at **213** (Bangladesh).
* **Chunk 1** covers **100–250** → ✅ answer found here.
* **Chunk 2** covers **200–300** → ✅ also contains answer.
* **Chunk 0** covers **0–150** → ❌ answer not here.

The loop makes sure every chunk “knows” if the answer belongs inside it.

---

### 📌 **Key takeaway**

* **`overflow_to_sample_mapping`** → links each **chunk** to its **original paragraph**.
* **`answer_start`** → gives the **character position** of the answer in the full text.
* The code compares the two (using `offset_mapping`) to locate the **token start & end** of the answer **for each chunk**.


In [46]:
max_length = 384 # used by google
stride = 128 

def tokenize_fun(batch):  # Changed parameter name from 'data' to 'batch'
    # Fix: batch already contains the questions, not data["train"]["question"]
    questions = [q.strip() for q in batch["question"]]  # Fixed: batch["question"] not data["train"]["question"]

    # Use the tokenizer to tokenize the questions and contexts
    inputs = tokenizer(
        questions,
        batch["context"],  # Fixed: batch["context"] not data["train"]["context"]
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length" # so that any context does not get truncated
    )

    offset_mapping = inputs.pop("offset_mapping")
    original_sample_mapping = inputs.pop("overflow_to_sample_mapping")

    answers = batch["answers"]  # Fixed: batch["answers"] not data["train"]["answers"]

    start_indices = []  # Fixed spelling
    end_indices = []    # Fixed spelling

    for i, offset_map in enumerate(offset_mapping):

        # Get the answer start and end character
        ans_containing_sample = original_sample_mapping[i] # this is the index of the sample in the original dataset
        # we do this so that due to tokenization, we can find the corresponding answer in the original dataset

        # From the answers, get the answer start character
        ans_start_char = answers[ans_containing_sample] 
        # ans_start_char is a dictionary with key "answer_start"
        ans_start_char = ans_start_char["answer_start"][0]
        ans_end_char = ans_start_char + len(answers[ans_containing_sample]["text"][0])
        
        sequence_ids = inputs.sequence_ids(i)

        context_start = sequence_ids.index(1)
        context_end = len(sequence_ids) - sequence_ids[::-1].index(1) - 1 

        start_idx, end_idx = find_answer_token_index(
            context_start,
            context_end,
            ans_start_char,
            ans_end_char,
            offset_map  # Fixed variable name
        )
        start_indices.append(start_idx)
        end_indices.append(end_idx)

    # Store the start and end indices in the inputs dictionary
    # these will be used to train the model
    inputs["start_positions"] = start_indices 
    inputs["end_positions"] = end_indices

    return inputs

#### 📝 Function Breakdown: `tokenize_fun`

**This function is the heart of our preprocessing!** It handles:

1. **Input Processing**: Clean questions, tokenize with context
2. **Overflow Handling**: Create multiple chunks for long contexts  
3. **Answer Alignment**: Map each chunk's answer positions
4. **Output Preparation**: Format data for model training

**Critical Details**:
- `batch["question"]` not `data["train"]["question"]` - we're processing the batch parameter
- `padding="max_length"` ensures consistent tensor sizes
- `remove_columns` eliminates original text data after tokenization
- Return `start_positions` and `end_positions` for training targets

In [47]:
data["train"]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 87599
})

In [48]:
train_dataset=data["train"].map(
    tokenize_fun,
    batched=True,
    remove_columns=data["train"].column_names # unnecessary columns will be removed from the dataset after tokenization
)


#### 🚀 Applying Tokenization to Training Data

Using `map()` with `batched=True` for efficient processing:
- Processes multiple examples at once (faster)
- Removes original columns we no longer need
- Creates new columns: `input_ids`, `attention_mask`, `start_positions`, `end_positions`

In [49]:
data["validation"][0]

{'id': '56be4db0acb8001400a502ec',
 'title': 'Super_Bowl_50',
 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.',
 'question': 'Which NFL team represented the AFC at Super Bowl 50?',
 'answers': {'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'],


In [50]:
def tokenize_fun_val(batch):
    
    questions = [q.strip() for q in batch["question"]] # remove leading and trailing whitespace
    inputs = tokenizer(
        questions,
        batch["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )
    original_sample_mapping = inputs.pop("overflow_to_sample_mapping")
    sample_ids = []

    # here we will rewrite the offset mapping by removing the tokens of question ids with None
    for i in range(len(inputs["input_ids"])):
        sample_id = original_sample_mapping[i]
        sample_ids.append(batch["id"][sample_id])  # Use batch["id"] to get the original sample ID
        
        sequence_ids = inputs.sequence_ids(i)
        offset_mapping = inputs["offset_mapping"][i]

        inputs["offset_mapping"][i] = [
            x if sequence_ids[j] == 1 else None for j, x in enumerate(offset_mapping)
            ]

    inputs["sample_id"] = sample_ids  # Store the sample IDs in the inputs dictionary
    return inputs

#### 🎯 Validation Set: Different Requirements

**Why a separate function for validation?**

**Training**: Needs `start_positions` and `end_positions` for learning
**Validation**: Needs different outputs for answer extraction:
- `sample_id`: To match predictions with original questions
- `offset_mapping`: To convert token positions back to text
- No target positions needed (we'll predict them!)

**Key Difference**: We set question tokens' offset mappings to `None` so we only extract answers from context.

In [51]:
validation_dataset=data["validation"].map(
    tokenize_fun_val,
    batched=True,
    remove_columns=data["validation"].column_names
)


# 4. Implement Metric Function: Evaluating Question Answering

### 📊 Understanding SQuAD Metrics

**SQuAD uses two key metrics**:
1. **Exact Match (EM)**: Percentage of predictions that match ground truth exactly
2. **F1 Score**: Token-level F1 between prediction and ground truth (handles partial matches)

**Why both metrics?**
- EM is strict: "Barack Obama" vs "Obama" = 0% match
- F1 is lenient: "Barack Obama" vs "Obama" = 50% match

**Input Format Required**:
- `predictions`: List of `{"id": "123", "prediction_text": "answer"}`
- `references`: List of `{"id": "123", "answers": {"text": ["answer1", "answer2"], "answer_start": [45, 67]}}`

In [52]:
from evaluate import load

metric = load("squad")

In [53]:
predicted=[
    {"id":"1","prediction_text":"This is a dummy answer"},
    {"id":"2","prediction_text":"This is another dummy answer"},
    {"id":"3","prediction_text":"This is yet another dummy answer"}
]

true_answers=[
    {"id":"1","answers":{"answer_start":[0],"text":["This is a dummy answer"]}},
    {"id":"2","answers":{"answer_start":[0],"text":["This is another dummy answer"]}},
    {"id":"3","answers":{"answer_start":[0],"text":["This is yet another dummy answer"]}}
]

metric.compute(
    predictions=predicted,
    references=true_answers
)

{'exact_match': 100.0, 'f1': 100.0}

#### 🧪 Testing the Metric Function

Let's test with dummy data to understand the expected format:

# 5. From Logits to Answer Conversion: The Final Challenge

### 🧠 Understanding Model Output

**What the model returns**:
- `start_logits`: Score for each token being the START of the answer
- `end_logits`: Score for each token being the END of the answer

**Our job**: Convert these scores to actual text answers!

**The Process**:
1. **Load pre-trained model**: Use already fine-tuned DistilBERT
2. **Get predictions**: Run model on validation data  
3. **Find best spans**: Combine start/end logits to find highest-scoring answer spans
4. **Extract text**: Use offset mappings to convert token positions back to text
5. **Evaluate**: Compare with ground truth using SQuAD metrics

In [69]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased-distilled-squad")

Our goal here is to take some sample validation and work on pretrained fined tuned model

In [71]:
trianed_checkpoint = "distilbert-base-uncased-distilled-squad"

In [72]:
tokenizer2=AutoTokenizer.from_pretrained(trianed_checkpoint) # keep the original tokenizer seperate for testing
old_tokenizer = tokenizer2
tokenizer=tokenizer2


C:\Users\Lenovo\AppData\Roaming\Python\Python311\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


#### 🏆 Using Pre-trained Model for Testing

We'll use `distilbert-base-uncased-distilled-squad` - already fine-tuned on SQuAD!
This lets us test our pipeline before training our own model.

In [76]:
from datasets import  load_dataset

small_validation_dataset =load_dataset("squad")["validation"].select(range(100))  # Select a smaller subset for validation

In [77]:
small_validation_processed=small_validation_dataset.map(
    tokenize_fun_val,
    batched=True,
    remove_columns=data["validation"].column_names
)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

#### 🎯 Preparing Small Validation Set

For testing, we'll use only 100 examples to speed up the process:

In [78]:
tokenizer=old_tokenizer  # Reset tokenizer to the original one for testing

In [79]:
import torch

small_model_inputs=small_validation_processed.remove_columns(["sample_id","offset_mapping"])
small_model_inputs.set_format(type="torch")

move all the tensors to gpu if exists

convet the dataset into a dictionary where the columns are key and its values are value

In [81]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

small_model_inputs_gpu={
    key: small_model_inputs[key].to(device) for key in small_model_inputs.column_names
}
small_model_inputs_gpu

{'input_ids': tensor([[  101,  2029,  5088,  ...,     0,     0,     0],
         [  101,  2029,  5088,  ...,     0,     0,     0],
         [  101,  2073,  2106,  ...,     0,     0,     0],
         ...,
         [  101,  2054,  2048,  ...,     0,     0,     0],
         [  101,  2040,  3743,  ...,     0,     0,     0],
         [  101,  2040, 25214,  ...,     0,     0,     0]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]])}

#### ⚡ GPU Acceleration Setup

Converting data to GPU format for faster inference:
- Remove non-tensor columns (`sample_id`, `offset_mapping`) 
- Move tensors to GPU if available
- Format as dictionary for model input

In [82]:
pretrained_model = AutoModelForQuestionAnswering.from_pretrained(trianed_checkpoint).to(device)  # Move the model to the GPU if available


In [83]:
with torch.no_grad():
    outputs = pretrained_model(**small_model_inputs_gpu)

#### 🔮 Getting Model Predictions

Running inference with `torch.no_grad()` for efficiency (no gradient computation needed):

In [84]:
outputs

QuestionAnsweringModelOutput(loss=None, start_logits=tensor([[ -5.7964,  -5.4623,  -6.7543,  ..., -10.0854, -10.0827, -10.0668],
        [ -5.2441,  -5.0187,  -6.3018,  ...,  -9.5495,  -9.5412,  -9.5359],
        [ -6.0572,  -5.9766,  -8.6033,  ..., -10.9921, -10.9982, -10.9832],
        ...,
        [ -4.2015,  -5.0836,  -5.7509,  ...,  -9.0691,  -9.0729,  -9.0472],
        [ -5.8249,  -5.5849,  -6.9967,  ..., -10.2358, -10.2125, -10.1955],
        [ -5.5046,  -5.2944,  -6.6587,  ...,  -9.8502,  -9.8481,  -9.8277]]), end_logits=tensor([[ -2.9106,  -5.2178,  -5.9634,  ..., -10.1748, -10.1949, -10.1774],
        [ -2.7664,  -4.9725,  -5.6371,  ...,  -9.8139,  -9.8348,  -9.8158],
        [ -2.5560,  -6.4450,  -7.2670,  ..., -10.6284, -10.6472, -10.6480],
        ...,
        [ -1.2792,  -6.0971,  -6.1631,  ...,  -9.3162,  -9.3135,  -9.3101],
        [ -2.5531,  -4.9517,  -6.1030,  ...,  -9.9603, -10.0066,  -9.9767],
        [ -1.0698,  -4.0969,  -6.1800,  ...,  -9.7161,  -9.7476,  -9.740

In [85]:
start_logits = outputs.start_logits.cpu().numpy()  # Move to CPU and convert to numpy array
end_logits = outputs.end_logits.cpu().numpy()  # Move to CPU and convert to numpy


In [87]:
small_validation_processed["sample_id"][:5]

['56be4db0acb8001400a502ec',
 '56be4db0acb8001400a502ed',
 '56be4db0acb8001400a502ee',
 '56be4db0acb8001400a502ef',
 '56be4db0acb8001400a502f0']

In [90]:
len(validation_dataset["sample_id"])

10784

In [91]:
len(set(validation_dataset["sample_id"]))

10570

#### ⚠️ Handling Duplicate Sample IDs

**Critical Issue**: Due to overflow tokenization, one original sample can create multiple processed chunks!

**Example**: 
- Original sample ID: "abc123"
- After tokenization: 3 chunks all with sample_id "abc123"

**Solution**: Map each sample_id to **all its chunk indices**, then check all chunks to find the best answer.

In [106]:
sample_id2idx={}

# Create a mapping from sample_id to index
for idx, sample_id in enumerate(small_validation_processed["sample_id"]):
    if sample_id not in sample_id2idx:
        sample_id2idx[sample_id] = idx 
    else:
        print(f"Duplicate sample_id found: {sample_id} at index {idx}")
        sample_id2idx[sample_id].appened(idx)  # Update to the latest index

In [107]:
# method to get the largest values index

(-start_logits[0]).argsort() # by negative values, we can get the largest values index at first position

array([ 46,  47,  57,  38,  43,  39,  45,  54,  50,  58,  49,  48,  42,
        40,  13,  35,  78,  56,  59,  70,  69,  44,  31,  53,  75,  41,
        65,  81,  62,  51,  66,  37,  64,  95, 101,  52,  15,  19,  24,
        27,  96,  28,  61,  71,  18,  87, 164,  76,   1,  29,  73,  84,
        68,  63, 150,  36,  55,  80,  12, 170,   0,  74,  67,  92,  17,
        32,  22,  21, 100, 102,  14,  34,  20, 168,  26,  72,  33,  98,
        97, 135, 105,  23,   3,  90,  86, 159,  89,  30,  79, 110, 103,
       152,   2,   5,  85,  16,   4,  77, 165, 106, 115, 131,  88,  93,
       158, 163, 149,  83, 109,   6,   8,  91,  60, 104, 107, 142, 143,
       124, 108,   7,  99, 129, 140, 123, 156, 136,  82,  25,  94, 111,
       118, 161,  10, 114, 144, 166, 169, 162, 130, 147, 151, 133, 127,
       112, 167, 132, 141,   9, 117, 119, 134, 138, 113,  11, 137, 148,
       120, 155, 139, 157, 153, 146, 160, 126, 121, 154, 145, 122, 125,
       116, 128, 239, 251, 240, 252, 248, 249, 237, 254, 224, 17

#### 🏆 Finding the Best Answer Spans

**Strategy for logit processing**:
1. Get indices of highest logits using `argsort()` with negative values
2. Consider top N candidates for both start and end positions  
3. Try all valid combinations (start < end, reasonable length)
4. Pick combination with highest combined score

In [108]:
start_logits[0][(-start_logits[0]).argsort()]

array([  9.212207  ,   4.0875874 ,   2.7158997 ,   2.2725255 ,
         0.39875352,  -0.40898734,  -0.6629695 ,  -1.2148442 ,
        -1.4789754 ,  -1.4856263 ,  -1.8724651 ,  -2.26385   ,
        -2.52184   ,  -3.0891404 ,  -3.1790962 ,  -3.3774514 ,
        -3.6189504 ,  -3.8023033 ,  -3.83256   ,  -3.8696363 ,
        -4.0021224 ,  -4.060046  ,  -4.1088333 ,  -4.155589  ,
        -4.2564936 ,  -4.2588897 ,  -4.3052034 ,  -4.3189926 ,
        -4.3593597 ,  -4.422491  ,  -4.4322224 ,  -4.501423  ,
        -4.519956  ,  -4.59059   ,  -4.8802023 ,  -5.0128536 ,
        -5.0463114 ,  -5.0923142 ,  -5.1064043 ,  -5.126451  ,
        -5.1896067 ,  -5.1984496 ,  -5.271161  ,  -5.40046   ,
        -5.426758  ,  -5.457922  ,  -5.460877  ,  -5.461216  ,
        -5.4622917 ,  -5.517152  ,  -5.5184627 ,  -5.591178  ,
        -5.59536   ,  -5.662509  ,  -5.714512  ,  -5.7590785 ,
        -5.765306  ,  -5.7858534 ,  -5.786115  ,  -5.786401  ,
        -5.7964277 ,  -5.8406816 ,  -5.9035983 ,  -5.91

In [109]:
small_validation_processed["offset_mapping"][0]

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 [0, 5],
 [6, 10],
 [11, 13],
 [14, 17],
 [18, 20],
 [21, 29],
 [30, 38],
 [39, 43],
 [44, 46],
 [47, 56],
 [57, 60],
 [61, 69],
 [70, 72],
 [73, 76],
 [77, 85],
 [86, 94],
 [95, 101],
 [102, 103],
 [103, 106],
 [106, 107],
 [108, 111],
 [112, 115],
 [116, 120],
 [121, 127],
 [127, 128],
 [129, 132],
 [133, 141],
 [142, 150],
 [151, 161],
 [162, 163],
 [163, 166],
 [166, 167],
 [168, 176],
 [177, 183],
 [184, 191],
 [192, 200],
 [201, 204],
 [205, 213],
 [214, 222],
 [223, 233],
 [234, 235],
 [235, 238],
 [238, 239],
 [240, 248],
 [249, 257],
 [258, 266],
 [267, 269],
 [269, 270],
 [270, 272],
 [273, 275],
 [276, 280],
 [281, 286],
 [287, 292],
 [293, 298],
 [299, 303],
 [304, 309],
 [309, 310],
 [311, 314],
 [315, 319],
 [320, 323],
 [324, 330],
 [331, 333],
 [334, 342],
 [343, 344],
 [344, 345],
 [346, 350],
 [350, 351],
 [352, 354],
 [355, 359],
 [359, 360],
 [360, 361],
 [362, 369],
 [370, 37

In [110]:
n_largest = 20
max_answer_length = 30
predicted_answers = []

for sample in small_validation_dataset:
    sample_id = sample["id"]
    context = sample["context"]

    best_score = -float("inf")
    best_answer = None

    # Find all indices for this sample_id in the processed dataset
    sample_indices = [i for i, sid in enumerate(small_validation_processed["sample_id"]) if sid == sample_id]
    
    # Check all chunks for this sample
    for idx in sample_indices:
        start_logits_sample = start_logits[idx]  # Fixed: use different variable name
        end_logits_sample = end_logits[idx]      # Fixed: use different variable name

        offset_mapping = small_validation_processed[idx]["offset_mapping"]

        start_indices = (-start_logits_sample).argsort()[:n_largest]  # Get indices of the n largest start logits
        end_indices = (-end_logits_sample).argsort()[:n_largest]      # Get indices of the n largest end logits
    
        for start_idx in start_indices:
            for end_idx in end_indices:
                if start_idx >= end_idx:
                    continue
                if end_idx - start_idx + 1 > max_answer_length:
                    continue
                if offset_mapping[start_idx] is None or offset_mapping[end_idx] is None:
                    continue

                score = start_logits_sample[start_idx] + end_logits_sample[end_idx]
                if score > best_score:
                    best_score = score
                    start_char, end_char = offset_mapping[start_idx][0], offset_mapping[end_idx][1]  # Fixed: get start and end chars correctly
                    best_answer = context[start_char:end_char]
                
    if best_answer is not None:
        predicted_answers.append({
            "id": sample_id,
            "prediction_text": best_answer  # Fixed: was best_ans
        })
    else:
        # If no valid answer found, add empty prediction
        predicted_answers.append({
            "id": sample_id,
            "prediction_text": ""
        })

print(f"Generated {len(predicted_answers)} predictions")
predicted_answers[:3]  # Show first 3 predictions

Generated 100 predictions


[{'id': '56be4db0acb8001400a502ec', 'prediction_text': 'Denver Broncos'},
 {'id': '56be4db0acb8001400a502ed', 'prediction_text': 'Carolina Panthers'},
 {'id': '56be4db0acb8001400a502ee', 'prediction_text': "Levi's Stadium"}]

### 🔥 **The Most Challenging Part: Answer Prediction Logic**

This code is complex because it handles **multiple edge cases** that occur in real-world QA:

#### 🧩 **The Problem**:
- One question might be split into **multiple chunks** (due to long context)
- Each chunk has its own **start/end logits** 
- We need to find the **best answer across ALL chunks**

#### 🛠️ **The Solution Strategy**:

1. **📋 Outer Loop**: `for sample in small_validation_dataset`
   - Process each original question one by one

2. **🔍 Find All Chunks**: `sample_indices = [i for i, sid in enumerate(...) if sid == sample_id]`  
   - One question → multiple processed chunks
   - Find ALL chunks belonging to this question

3. **🎯 Inner Loop**: `for idx in sample_indices`
   - Check each chunk separately for potential answers

4. **🏆 Score All Combinations**: Nested loops `for start_idx in start_indices: for end_idx in end_indices:`
   - Try top 20 start positions × top 20 end positions = 400 combinations per chunk!
   - Apply constraints: `start < end`, `reasonable length`, `valid offset mapping`

5. **💯 Keep the Best**: `if score > best_score:`
   - Track the highest-scoring answer across ALL chunks
   - Convert token indices → character positions → actual text

#### ⚠️ **Why It's Tricky**:
- **Chunk Management**: Same question split across multiple chunks
- **Logit Processing**: Converting probability scores to text spans
- **Constraint Validation**: Many combinations are invalid
- **Offset Mapping**: Token positions → character positions → text extraction

#### 🎯 **Key Variables**:
- `sample_indices`: All chunk indices for current question
- `start_logits_sample[idx]`: Start probabilities for chunk `idx`  
- `offset_mapping[start_idx][0]`: Character position of token `start_idx`
- `context[start_char:end_char]`: Final extracted answer text

#### 🎯 The Complete Answer Extraction Pipeline

**This is the most complex part!** For each original sample:

1. **Find all chunks**: Get all processed chunks belonging to this sample
2. **Score all spans**: For each chunk, try top N start/end position combinations
3. **Validate spans**: Check constraints (start < end, reasonable length, valid offsets)
4. **Extract text**: Use offset mapping to get character positions, then slice original context
5. **Keep best**: Track highest-scoring answer across all chunks

**Key Parameters**:
- `n_largest=20`: Consider top 20 start/end positions
- `max_answer_length=30`: Reasonable upper bound for answer length

In [111]:
true_answers = [
    {"id":x["id"], "answers": x["answers"]} for x in small_validation_dataset
]

In [112]:
metric.compute(
    predictions=predicted_answers,
    references=true_answers
)

{'exact_match': 58.0, 'f1': 72.8639194139194}

#### 📊 Evaluating Our Pipeline

Let's see how well our extraction pipeline works with the pre-trained model!

In [ ]:
from tqdm.autonotebook import tqdm

def compute_metrics(start_logits, end_logits, processed_dataset, original_dataset):
    """
    Extract answers from model logits and compute SQuAD metrics.
    
    Args:
        start_logits: Array of start position scores for each token
        end_logits: Array of end position scores for each token  
        processed_dataset: Tokenized dataset with offset mappings
        original_dataset: Original dataset with questions and contexts
    
    Returns:
        Dictionary with exact_match and f1 scores
    """
    # Configuration
    TOP_N_CANDIDATES = 20      # Consider top N start/end positions
    MAX_ANSWER_LENGTH = 30     # Maximum tokens in answer span
    
    # Step 1: Build mapping from sample_id to all its chunk indices
    sample_to_chunks = {}
    for chunk_idx, sample_id in enumerate(processed_dataset["sample_id"]):
        if sample_id not in sample_to_chunks:
            sample_to_chunks[sample_id] = []
        sample_to_chunks[sample_id].append(chunk_idx)
    
    # Step 2: Process each original sample to find best answer
    predictions = []
    
    for sample in tqdm(original_dataset, desc="Extracting answers"):
        sample_id = sample["id"]
        context = sample["context"]
        
        # Initialize tracking variables
        best_score = -float("inf")
        best_answer_text = ""
        
        # Get all chunks for this sample
        chunk_indices = sample_to_chunks.get(sample_id, [])
        if not chunk_indices:
            predictions.append({"id": sample_id, "prediction_text": ""})
            continue
            
        # Step 3: Check each chunk for potential answers
        for chunk_idx in chunk_indices:
            chunk_start_logits = start_logits[chunk_idx]
            chunk_end_logits = end_logits[chunk_idx]
            chunk_offset_mapping = processed_dataset[chunk_idx]["offset_mapping"]
            
            # Get top candidate positions
            top_start_positions = (-chunk_start_logits).argsort()[:TOP_N_CANDIDATES]
            top_end_positions = (-chunk_end_logits).argsort()[:TOP_N_CANDIDATES]
            
            # Step 4: Try all valid start/end combinations
            for start_pos in top_start_positions:
                for end_pos in top_end_positions:
                    # Apply constraints
                    if not _is_valid_span(start_pos, end_pos, chunk_offset_mapping, MAX_ANSWER_LENGTH):
                        continue
                    
                    # Calculate combined score
                    span_score = chunk_start_logits[start_pos] + chunk_end_logits[end_pos]
                    
                    # Update best answer if this score is higher
                    if span_score > best_score:
                        best_score = span_score
                        best_answer_text = _extract_answer_text(
                            start_pos, end_pos, chunk_offset_mapping, context
                        )
        
        # Add prediction for this sample
        predictions.append({
            "id": sample_id,
            "prediction_text": best_answer_text
        })
    
    # Step 5: Format ground truth and compute metrics
    references = [
        {"id": sample["id"], "answers": sample["answers"]} 
        for sample in original_dataset
    ]
    
    return metric.compute(predictions=predictions, references=references)


def _is_valid_span(start_pos, end_pos, offset_mapping, max_length):
    """Check if a start/end position pair forms a valid answer span."""
    # Start must come before end
    if start_pos >= end_pos:
        return False
    
    # Answer shouldn't be too long
    if end_pos - start_pos + 1 > max_length:
        return False
    
    # Both positions must have valid offset mappings (context tokens only)
    if offset_mapping[start_pos] is None or offset_mapping[end_pos] is None:
        return False
    
    return True


def _extract_answer_text(start_pos, end_pos, offset_mapping, context):
    """Extract answer text from context using offset mapping."""
    start_char = offset_mapping[start_pos][0] 
    end_char = offset_mapping[end_pos][1]
    return context[start_char:end_char]

#### 🔧 Final Reusable Function: `compute_metrics`

This function encapsulates our entire answer extraction pipeline:
- Handles sample ID to index mapping (including duplicates)
- Processes all chunks for each sample
- Finds best answer spans across all chunks
- Returns formatted results for SQuAD evaluation

**Use this function** to evaluate any QA model's predictions!

In [118]:
compute_metrics(start_logits, end_logits, small_validation_processed, small_validation_dataset)

Processing samples:   0%|          | 0/100 [00:00<?, ?it/s]

{'exact_match': 58.0, 'f1': 72.8639194139194}

In [119]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="distilbert-finetuned-squad",
    evaluation_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    fp16=True,  # Enable mixed precision training
    weight_decay=0.01,
)

C:\Users\Lenovo\AppData\Roaming\Python\Python311\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


# 6. Model Training: Fine-tuning DistilBERT

### 🏋️ Training Configuration

**Key training parameters**:
- `learning_rate=2e-5`: Standard for BERT fine-tuning
- `num_train_epochs=3`: Usually sufficient for QA
- `fp16=True`: Mixed precision for faster training
- `weight_decay=0.01`: Regularization to prevent overfitting

**Why these values?** Based on extensive research and empirical results from the BERT paper and SQuAD leaderboards.

In [120]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    tokenizer=tokenizer,
)
trainer.train()

  0%|          | 0/33198 [00:00<?, ?it/s]

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


KeyboardInterrupt: 

#### 🚀 Starting the Training Process

The Trainer handles all the complexity:
- **Loss calculation**: Automatically computes cross-entropy loss for start/end positions
- **Optimization**: Adam optimizer with learning rate scheduling
- **Checkpointing**: Saves model after each epoch
- **Progress tracking**: Shows training progress and metrics

In [121]:
trainer_output = trainer.predict(validation_dataset)
type(trainer_output)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/1348 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
predictions, _= trainer_output  # Unpack predictions and metrics

In [ ]:
# save model
# import pipeline and start using it